# Facial Landmark CSV Visualization (Notebook Version)

This notebook:
- Loads a landmarks CSV (MediaPipe-like `x_i, y_i, z_i` columns).
- Animates the landmarks as scatter + optional mesh (no background face).
- Plots **other numeric columns** as **one-column-per-image** line graphs (min–max normalized).
- Lets you tweak configuration in cells without editing a CLI script.

> Tip: GPU is **not** needed for this notebook. Everything is CPU-bound (pandas / matplotlib / ffmpeg).


Set the input CSV/video paths, output folder, frame rate, and plotting options before running later cells.


In [8]:
# =======================
# CONFIGURATION
# =======================
CSV_PATH = "./still_out/SFE_multi_true.csv"   # multi-face CSV
OUTDIR = "./vis_still_multi"                  # where to save figures/animations
FPS = 30
MAX_FRAMES = 0                                 # 0 = use all frames
DRAW_MESH = True
MAX_SERIES_PER_LEGEND = 20
NORMALIZE_TIMESERIES = True
SPLIT_BATCH_SIZE = 20

# Optional: limit which non-landmark columns to plot (None = all)
INCLUDE_COLUMNS = None
EXCLUDE_COLUMNS = None

# Reproducibility / sampling
RANDOM_SEED = 123


Set the input CSV/video paths, output folder, frame rate, and plotting options before running later cells.


In [9]:
# Imports
import os, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FFMpegWriter
from pathlib import Path

# Create output dir
outdir = Path(OUTDIR)
outdir.mkdir(parents=True, exist_ok=True)

# # Helpers for pretty display
# from caas_jupyter_tools import display_dataframe_to_user

print(f"Using CSV: {CSV_PATH}\nSaving to: {outdir}")

Using CSV: ./still_out/SFE_multi_true.csv
Saving to: vis_still_multi


Load the multi-face CSV, normalize frame/face indexing, and group rows so each frame can render all detected faces.


In [10]:
# Load CSV & preview (multi-face long format)
df = pd.read_csv(CSV_PATH)
if "Frame" not in df.columns:
    df["Frame"] = np.arange(len(df), dtype=int)
if "Face_Index" not in df.columns:
    df["Face_Index"] = 0

# Ensure stable ordering by frame then face index
df["Frame"] = pd.to_numeric(df["Frame"], errors="coerce").fillna(-1).astype(int)
df["Face_Index"] = pd.to_numeric(df["Face_Index"], errors="coerce").fillna(-1).astype(int)
df = df.sort_values(["Frame", "Face_Index"]).reset_index(drop=True)

# Group rows by frame for multi-face rendering
frame_groups = {int(fid): grp.reset_index(drop=True) for fid, grp in df.groupby("Frame", sort=True)}
frame_ids = sorted(frame_groups.keys())

print(f"DataFrame shape: {df.shape}")
print(f"Unique frames: {len(frame_ids)}")
print(f"Max faces in a frame: {max(len(g) for g in frame_groups.values()) if frame_groups else 0}")


DataFrame shape: (4458, 1420)
Unique frames: 3188
Max faces in a frame: 3


Find the MediaPipe landmark triplets (`x_i`, `y_i`, `z_i`) so the notebook can separate landmarks from other numeric signals.


In [11]:
# Detect landmark columns forming triplets (x_i, y_i, z_i)

def detect_landmark_columns(df):
    patterns = [
        (re.compile(r"^x[_]?(\d+)$"), re.compile(r"^y[_]?(\d+)$"), re.compile(r"^z[_]?(\d+)$")),
        (re.compile(r"^x_(\d+)$"), re.compile(r"^y_(\d+)$"), re.compile(r"^z_(\d+)$")),
        (re.compile(r"^landmark[_-]?(\d+)_x$"), re.compile(r"^landmark[_-]?(\d+)_y$"), re.compile(r"^landmark[_-]?(\d+)_z$")),
        (re.compile(r"^x\[(\d+)\]$"), re.compile(r"^y\[(\d+)\]$"), re.compile(r"^z\[(\d+)\]$")),
    ]
    x_cols, y_cols, z_cols = {}, {}, {}
    for col in df.columns:
        col_s = col.strip()
        matched = False
        for rx, ry, rz in patterns:
            if rx.match(col_s):
                i = int(rx.match(col_s).group(1)); x_cols[i] = col; matched = True; break
            if ry.match(col_s):
                i = int(ry.match(col_s).group(1)); y_cols[i] = col; matched = True; break
            if rz.match(col_s):
                i = int(rz.match(col_s).group(1)); z_cols[i] = col; matched = True; break
        if not matched:
            continue

    # fallback: generic x_#, y_#, z_#
    if not (set(x_cols) and set(y_cols) and set(z_cols)):
        rx = re.compile(r"^x[_-]?(\d+)$", re.IGNORECASE)
        ry = re.compile(r"^y[_-]?(\d+)$", re.IGNORECASE)
        rz = re.compile(r"^z[_-]?(\d+)$", re.IGNORECASE)
        for col in df.columns:
            m = rx.match(col);  m and x_cols.setdefault(int(m.group(1)), col)
            m = ry.match(col);  m and y_cols.setdefault(int(m.group(1)), col)
            m = rz.match(col);  m and z_cols.setdefault(int(m.group(1)), col)

    idxs = sorted(set(x_cols.keys()) & set(y_cols.keys()) & set(z_cols.keys()))
    return idxs, x_cols, y_cols, z_cols

idxs, x_cols, y_cols, z_cols = detect_landmark_columns(df)
print(f"Detected landmarks: {len(idxs)}")

Detected landmarks: 468


Prepare helper functions for drawing landmark scatter points and optional mesh connections.


In [12]:
# Triangulation helper

def _row_xy(row, idxs, x_cols, y_cols):
    xs = row[[x_cols[i] for i in idxs]].to_numpy(dtype=float)
    ys = row[[y_cols[i] for i in idxs]].to_numpy(dtype=float)
    ok = np.isfinite(xs) & np.isfinite(ys)
    if not np.any(ok):
        return None, None
    xs = xs[ok]
    ys = ys[ok]
    if len(xs) < 3:
        return None, None
    pts = np.column_stack([xs, ys])
    if len(np.unique(pts, axis=0)) < 3:
        return None, None
    return xs, ys


def base_triangulation(df, idxs, x_cols, y_cols, draw_mesh=True):
    if not draw_mesh or len(idxs) < 3:
        return None

    for _, row in df.iterrows():
        xs, ys = _row_xy(row, idxs, x_cols, y_cols)
        if xs is None:
            continue
        y_disp = 1.0 - ys
        return mtri.Triangulation(xs, y_disp)

    return None

triang = base_triangulation(df, idxs, x_cols, y_cols, draw_mesh=DRAW_MESH)
triang is not None


True

### This cell outputs the landmark on xy axis, without original video

Prepare helper functions for drawing landmark scatter points and optional mesh connections.


In [ ]:
# Animate landmarks (scatter + optional mesh) for multi-face CSV
mp4_path = outdir / "SFE_multi.mp4"

n_frames = len(frame_ids)
step = max(1, n_frames // MAX_FRAMES) if MAX_FRAMES else 1
frames_idx = list(range(0, n_frames, step))

fig = plt.figure(figsize=(12, 9))
ax = plt.gca()
ax.set_aspect("auto")
ax.set_title("Facial Landmarks (Multi-face)")

scatter = ax.scatter([], [], s=8)
holders = {"mesh": None, "bbox": None}

def update_frame(frame_pos):
    frame_id = frame_ids[frame_pos]
    g = frame_groups[frame_id]

    xs_all = []
    ys_all = []

    if holders["mesh"] is not None:
        for coll in holders["mesh"]:
            coll.remove()
        holders["mesh"] = None

    mesh_artists = []
    for _, row in g.iterrows():
        xs, ys = _row_xy(row, idxs, x_cols, y_cols)
        if xs is None:
            continue
        ys_disp = 1.0 - ys
        xs_all.append(xs)
        ys_all.append(ys_disp)

        if triang is not None and len(xs) == len(triang.x):
            tri = mtri.Triangulation(xs, ys_disp, triangles=triang.triangles)
            mesh_artists.extend(ax.triplot(tri, linewidth=0.5))

    if xs_all:
        X = np.concatenate(xs_all)
        Y = np.concatenate(ys_all)
        scatter.set_offsets(np.c_[X, Y])

        xmin, xmax = float(np.nanmin(X)), float(np.nanmax(X))
        ymin, ymax = float(np.nanmin(Y)), float(np.nanmax(Y))
        pad = 0.02
        xmin, xmax = max(0, xmin - pad), min(1, xmax + pad)
        ymin, ymax = max(0, ymin - pad), min(1, ymax + pad)
        if holders["bbox"] is not None:
            holders["bbox"].remove()
        holders["bbox"] = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, linewidth=1.0)
        ax.add_patch(holders["bbox"])
    else:
        scatter.set_offsets(np.empty((0, 2)))
        if holders["bbox"] is not None:
            holders["bbox"].remove()
            holders["bbox"] = None

    holders["mesh"] = mesh_artists if mesh_artists else None
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

writer = FFMpegWriter(fps=FPS, metadata=dict(artist="landmark_viz_notebook"), bitrate=1800)
with writer.saving(fig, str(mp4_path), dpi=150):
    for idx in frames_idx:
        update_frame(idx)
        writer.grab_frame()

plt.close(fig)
print(f"Saved animation to: {mp4_path}")


Saved animation to: vis_still_multi/MRE_multi.mp4


### This cell outputs the video with facial landmark overlay

Import the analysis and plotting libraries, create the output directory, and confirm where files will be saved.


In [13]:
# --- PIXEL-ACCURATE OVERLAY (video background + landmark scatter + mesh) for multi-face ---

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FFMpegWriter
from pathlib import Path

# ========= CONFIG =========
VIDEO_PATH = "./still_in/Still face experiment - Lise-Lotte Austad (720p, h264).mp4"
OUT_VIDEO  = outdir / "SFE_overlay_multi.mp4"

USE_VIDEO_FPS  = True
POINT_SIZE     = 8
MESH_LINEWIDTH = 0.5

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

vid_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vid_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
total_frames_df = len(frame_ids)

n_frames = min(total_frames_video, total_frames_df)
if n_frames == 0:
    cap.release()
    raise RuntimeError("No frames available to overlay (check video and CSV).")

out_fps = vid_fps if (USE_VIDEO_FPS and vid_fps > 0) else FPS

print(f"Video size: {vid_w}x{vid_h}, frames: {total_frames_video}, using {n_frames} frames")
print(f"Output FPS: {out_fps}")
print(f"Saving overlay video to: {OUT_VIDEO}")

ret, frame0 = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Failed to read first frame from video.")

frame0_rgb = cv2.cvtColor(frame0, cv2.COLOR_BGR2RGB)

fig = plt.figure(figsize=(12, 9))
ax = plt.gca()
ax.set_title("Video with Facial Landmarks (Multi-face)")
img_artist = ax.imshow(frame0_rgb, extent=(0, vid_w, vid_h, 0), origin="upper")
scatter = ax.scatter([], [], s=POINT_SIZE)

holders = {"mesh": None}
triangles = triang.triangles if (triang is not None) else None

def update_frame(frame_pos, frame_bgr):
    frame_id = frame_ids[frame_pos]
    g = frame_groups[frame_id]

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img_artist.set_data(frame_rgb)

    if holders["mesh"] is not None:
        for coll in holders["mesh"]:
            coll.remove()
        holders["mesh"] = None

    xs_all = []
    ys_all = []
    mesh_artists = []

    for _, row in g.iterrows():
        xs, ys = _row_xy(row, idxs, x_cols, y_cols)
        if xs is None:
            continue

        xp = xs * vid_w
        yp = ys * vid_h
        xs_all.append(xp)
        ys_all.append(yp)

        if triangles is not None and len(xs) == len(triang.x):
            tri_pix = mtri.Triangulation(xp, yp, triangles=triangles)
            mesh_artists.extend(ax.triplot(tri_pix, linewidth=MESH_LINEWIDTH))

    if xs_all:
        X = np.concatenate(xs_all)
        Y = np.concatenate(ys_all)
        scatter.set_offsets(np.c_[X, Y])
    else:
        scatter.set_offsets(np.empty((0, 2)))

    holders["mesh"] = mesh_artists if mesh_artists else None
    ax.set_xlim(0, vid_w)
    ax.set_ylim(vid_h, 0)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
writer = FFMpegWriter(fps=out_fps, metadata=dict(artist="landmark_viz_notebook"), bitrate=1800)

with writer.saving(fig, str(OUT_VIDEO), dpi=150):
    for i in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break
        update_frame(i, frame)
        writer.grab_frame()

cap.release()
plt.close(fig)
print(f"Saved pixel-accurate overlay video to: {OUT_VIDEO}")


Video size: 1280x720, frames: 3191, using 3188 frames
Output FPS: 29.97002997002997
Saving overlay video to: vis_still_multi/SFE_overlay_multi.mp4
Saved pixel-accurate overlay video to: vis_still_multi/SFE_overlay_multi.mp4


Select numeric columns that are not landmark coordinates, such as frame numbers, pose, aspect ratios, and AU scores.


In [23]:
# Identify non-landmark numeric columns
lm_cols = {x_cols[i] for i in idxs} | {y_cols[i] for i in idxs} | {z_cols[i] for i in idxs}
numeric_cols = [c for c in df.columns if c not in lm_cols and pd.api.types.is_numeric_dtype(df[c])]

# Optional include/exclude filters
if INCLUDE_COLUMNS is not None:
    numeric_cols = [c for c in numeric_cols if c in set(INCLUDE_COLUMNS)]
if EXCLUDE_COLUMNS is not None:
    numeric_cols = [c for c in numeric_cols if c not in set(EXCLUDE_COLUMNS)]

print(f"Total non-landmark numeric columns: {len(numeric_cols)}")
numeric_cols[:10]

Total non-landmark numeric columns: 24


['Unnamed: 0',
 'Frame',
 'Pitch',
 'Yaw',
 'Roll',
 'Eye Aspect Ratio',
 'Mouth Aspect Ratio',
 'Occ_au_1',
 'Occ_au_2',
 'Occ_au_4']

Normalize selected time-series columns so signals with different scales can be plotted together or compared visually.


In [24]:
# Prepare a normalized per-frame copy for plotting
# Multi-face CSV has multiple rows per frame; aggregate numeric columns by frame.
df_frame = df.groupby("Frame", sort=True).mean(numeric_only=True)
ts = df_frame[numeric_cols].copy()

if NORMALIZE_TIMESERIES:
    for c in numeric_cols:
        s = ts[c].astype(float)
        mn = np.nanmin(s)
        mx = np.nanmax(s)
        if np.isfinite(mn) and np.isfinite(mx) and mx != mn:
            ts[c] = (s - mn) / (mx - mn)

print("Prepared time series (normalized)" if NORMALIZE_TIMESERIES else "Prepared time series (raw)")


Prepared time series (normalized)


Save one PNG line plot per selected numeric column.


In [25]:
# Plot each numeric column as ONE image (one-column-per-image), batched to avoid huge loops in one cell.

def plot_one_column(cname, outdir):
    fig = plt.figure(figsize=(10, 3.5))
    ax = plt.gca()
    ax.plot(ts.index.values, ts[cname].values, linewidth=1.0)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Normalized value" if NORMALIZE_TIMESERIES else "Value")
    ax.set_title(cname)
    fig.tight_layout()
    out_path = Path(outdir) / f"ts_{cname}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path

# Batch over columns
saved = []
for i, cname in enumerate(numeric_cols):
    p = plot_one_column(cname, outdir)
    saved.append(str(p))
    if (i+1) % max(1, SPLIT_BATCH_SIZE) == 0:
        print(f"...saved {i+1}/{len(numeric_cols)}")

print(f"Saved {len(saved)} one-column images to {outdir}")

...saved 20/24
Saved 24 one-column images to visualize_adult


Print a small sample of generated plot files to quickly confirm the output folder contents.


In [ ]:
# List a few generated per-column images
from itertools import islice
print("Sample outputs:")
for p in islice(sorted(outdir.glob("ts_*.png")), 10):
    print(" -", p)

In [28]:
from pathlib import Path

# columns that are NOT landmarks and are numeric
lm_cols = {x_cols[i] for i in idxs} | {y_cols[i] for i in idxs} | {z_cols[i] for i in idxs}
numeric_cols = [
    c for c in df.columns
    if c not in lm_cols and pd.api.types.is_numeric_dtype(df[c])
]

# optional include/exclude filters if you were using them
if INCLUDE_COLUMNS:
    numeric_cols = [c for c in numeric_cols if any(inc in c for inc in INCLUDE_COLUMNS)]
if EXCLUDE_COLUMNS:
    numeric_cols = [c for c in numeric_cols if all(exc not in c for exc in EXCLUDE_COLUMNS)]

print(f"{len(numeric_cols)} numeric non-landmark columns")

# build time-series DataFrame aligned with df rows
ts = df[numeric_cols].copy()

if NORMALIZE_TIMESERIES:
    for c in numeric_cols:
        s = ts[c].astype(float)
        mn = np.nanmin(s)
        mx = np.nanmax(s)
        if np.isfinite(mn) and np.isfinite(mx) and mx != mn:
            ts[c] = (s - mn) / (mx - mn)

print("ts shape:", ts.shape)
ts_au_cols = [x for x in ts.columns if "au" in x]
ts = ts.loc[:, ts_au_cols]
print("ts shape:", ts.shape)
print(ts.columns)


16 numeric non-landmark columns
ts shape: (908, 16)
ts shape: (908, 9)
Index(['Occ_au_1', 'Occ_au_2', 'Occ_au_3', 'Occ_au_4', 'Occ_au_6', 'Occ_au_9',
       'Occ_au_12', 'Occ_au_20', 'Occ_au_28'],
      dtype='object')


In [37]:
# --- VIDEO + FACIAL LANDMARKS + TIME-SERIES WITH MOVING TICKER (multi-face aware) ---

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FFMpegWriter
from pathlib import Path
import math

VIDEO_PATH = "./still_in/MotorVideo_ReachToEat.mp4"
OUT_VIDEO  = outdir / "video_with_landmarks_and_ts_multi.mp4"

USE_VIDEO_FPS   = True
POINT_SIZE      = 8
MESH_LINEWIDTH  = 0.5
MAX_TS_ROWS     = 4

CROP_ENABLED = False
CROP_X0 = 0
CROP_X1 = 1000
CROP_Y0 = 0
CROP_Y1 = 1000

plot_cols = ts.columns
n_ts = len(plot_cols)
if n_ts == 0:
    raise RuntimeError("No numeric columns selected for time-series plotting.")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

vid_w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vid_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

n_frames = min(total_frames_video, len(frame_ids))
if n_frames == 0:
    cap.release()
    raise RuntimeError("No frames available to overlay (check video and CSV).")

out_fps = vid_fps if (USE_VIDEO_FPS and vid_fps > 0) else FPS

if CROP_ENABLED:
    crop_x0 = max(0, CROP_X0); crop_y0 = max(0, CROP_Y0)
    crop_x1 = min(vid_w, CROP_X1); crop_y1 = min(vid_h, CROP_Y1)
else:
    crop_x0, crop_y0, crop_x1, crop_y1 = 0, 0, vid_w, vid_h
crop_w = crop_x1 - crop_x0
crop_h = crop_y1 - crop_y0

ret, frame0 = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Failed to read first frame from video.")

frame0_rgb = cv2.cvtColor(frame0, cv2.COLOR_BGR2RGB)
frame0_rgb_disp = frame0_rgb[crop_y0:crop_y1, crop_x0:crop_x1]

n_rows = min(MAX_TS_ROWS, n_ts)
n_ts_cols = math.ceil(n_ts / n_rows)
fig_height = max(9, 2.8 * n_rows)
fig = plt.figure(figsize=(24, fig_height))

gs = fig.add_gridspec(nrows=n_rows, ncols=n_ts_cols + 1, width_ratios=[1.4] + [1] * n_ts_cols)
gs.update(hspace=0.4, wspace=0.15)

ax_vid = fig.add_subplot(gs[:, 0])
ax_vid.axis('off')
img_artist = ax_vid.imshow(frame0_rgb_disp, extent=(0, crop_w, crop_h, 0), origin='upper')
scatter = ax_vid.scatter([], [], s=POINT_SIZE)
holders = {"mesh": None}
triangles = triang.triangles if (triang is not None) else None
ax_vid.set_xlim(0, crop_w)
ax_vid.set_ylim(crop_h, 0)

ts_vlines = []
x_vals = ts.index.values

for k, cname in enumerate(plot_cols):
    row = k % n_rows
    col = k // n_rows
    ax_ts = fig.add_subplot(gs[row, col + 1])
    y_vals = ts[cname].values.astype(float)
    ax_ts.plot(x_vals, y_vals, linewidth=1.0)
    ax_ts.set_title(cname, fontsize=8)
    vline = ax_ts.axvline(x_vals[0], color='r', linestyle='--', linewidth=1.0)
    ax_ts.tick_params(labelsize=7)
    ts_vlines.append(vline)

def update_frame(frame_pos, frame_bgr):
    frame_id = frame_ids[frame_pos]
    g = frame_groups[frame_id]

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_rgb_disp = frame_rgb[crop_y0:crop_y1, crop_x0:crop_x1]
    img_artist.set_data(frame_rgb_disp)

    if holders["mesh"] is not None:
        for coll in holders["mesh"]:
            coll.remove()
        holders["mesh"] = None

    xs_all = []
    ys_all = []
    mesh_artists = []
    for _, row in g.iterrows():
        xs, ys = _row_xy(row, idxs, x_cols, y_cols)
        if xs is None:
            continue
        xp = xs * vid_w - crop_x0
        yp = ys * vid_h - crop_y0
        xs_all.append(xp)
        ys_all.append(yp)
        if triangles is not None and len(xs) == len(triang.x):
            tri_pix = mtri.Triangulation(xp, yp, triangles=triangles)
            mesh_artists.extend(ax_vid.triplot(tri_pix, linewidth=MESH_LINEWIDTH))

    if xs_all:
        X = np.concatenate(xs_all)
        Y = np.concatenate(ys_all)
        scatter.set_offsets(np.c_[X, Y])
    else:
        scatter.set_offsets(np.empty((0, 2)))

    holders["mesh"] = mesh_artists if mesh_artists else None

    x_curr = frame_id
    for vline in ts_vlines:
        vline.set_xdata([x_curr, x_curr])

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
writer = FFMpegWriter(fps=out_fps, metadata=dict(artist='landmark_viz_notebook'), bitrate=1800)
with writer.saving(fig, str(OUT_VIDEO), dpi=150):
    for i in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break
        update_frame(i, frame)
        writer.grab_frame()

cap.release()
plt.close(fig)
print(f"Saved combined video to: {OUT_VIDEO}")


Plotting 9 time-series on the right.
Video size: 1620x1080, frames: 908, using 908 frames
Output FPS: 30.00330432867056
Saving combined video to: visualize_infant/video_with_landmarks_and_ts.mp4
Using crop box x:[550,1150), y:[250,950) -> 600x700
Saved combined video to: visualize_infant/video_with_landmarks_and_ts.mp4


In [19]:
print(vid_w, vid_h)

1620 1080
